# Gig Deal Hunter - Freelance Opportunity Value Analyzer

An agentic AI system that scans a freelance gig feed, estimates the **fair market budget** for each project using a multi-agent ensemble, and surfaces the gigs where the client's posted budget is at or above market rate - giving freelancers the best-value opportunities.

**Architecture mirrors week 8 "The Price is Right":**

| Week 8 | This notebook |
|---|---|
| `SpecialistAgent` (fine-tuned LLM on Modal) | `GigSpecialistAgent` (cheap model with few-shot expert prompting) |
| `FrontierAgent` (GPT + ChromaDB RAG) | `GigFrontierAgent` (OpenRouter LLM + ChromaDB RAG) |
| `EnsembleAgent` | `GigEnsembleAgent` |
| `ScannerAgent` (RSS + LLM extraction) | `GigScannerAgent` (gig feed + LLM selection) |
| `PlanningAgent` | `GigPlannerAgent` |

In [ ]:
# %pip install -q openai python-dotenv chromadb sentence-transformers gradio

In [ ]:
import os
import logging
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Optional
import chromadb
from sentence_transformers import SentenceTransformer
import gradio as gr

from prompts.prompts import (
    SCANNER_SYSTEM, SCANNER_USER_PREFIX, SCANNER_USER_SUFFIX,
    SPECIALIST_SYSTEM, SPECIALIST_FEW_SHOTS,
    FRONTIER_SYSTEM, frontier_user,
)
from helper import extract_price, parse_json_safe

load_dotenv(override=True)
logging.basicConfig(level=logging.INFO, format="%(message)s")

# Single cheap model for everything — keeps costs minimal
MODEL = "qwen/qwen-2.5-7b-instruct"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
print("OpenRouter client ready")

## Building the Gig Corpus

We need two datasets:
- **`GIG_CORPUS`** - ~50 historical gigs with known fair budgets. Loaded into ChromaDB to power the RAG pipeline.
- **`GIG_FEED`** - 15 new gig postings with posted budgets (some above market, some below). This is our "RSS feed".

In [ ]:
# Historical gigs — fair_budget is ground-truth market rate
GIG_CORPUS = [
    {"description": "Build a responsive React.js dashboard with authentication, charts, and user management", "fair_budget": 1200},
    {"description": "Develop a Python FastAPI backend with PostgreSQL, Redis caching, and JWT auth", "fair_budget": 1500},
    {"description": "Create a Shopify store with custom theme, product setup, and payment integration", "fair_budget": 600},
    {"description": "Design a brand identity package: logo, colour palette, typography, and style guide", "fair_budget": 750},
    {"description": "Write 10 SEO blog articles (800 words each) for a B2B SaaS company", "fair_budget": 500},
    {"description": "Build a WordPress site with WooCommerce, 20 products, and custom checkout flow", "fair_budget": 700},
    {"description": "Develop a Flutter mobile app for iOS and Android with Firebase backend", "fair_budget": 3500},
    {"description": "Set up AWS infrastructure with EC2, RDS, S3, and CloudFront using Terraform", "fair_budget": 1800},
    {"description": "Build a machine learning model for sentiment analysis on customer reviews", "fair_budget": 1200},
    {"description": "Design 5 landing pages in Figma with mobile-responsive wireframes", "fair_budget": 900},
    {"description": "Build a web scraper for real-estate listings with scheduled runs and email alerts", "fair_budget": 450},
    {"description": "Develop a Discord bot with moderation commands, logging, and a levelling system", "fair_budget": 300},
    {"description": "Create an animated explainer video (60 seconds) with voiceover for a SaaS product", "fair_budget": 800},
    {"description": "Migrate a legacy PHP app to Laravel with test coverage and documentation", "fair_budget": 2200},
    {"description": "Build a data pipeline using Apache Spark and Airflow on GCP Dataproc", "fair_budget": 2500},
    {"description": "Write technical documentation and API reference for a Node.js REST API", "fair_budget": 400},
    {"description": "Configure and optimise a Kubernetes cluster on DigitalOcean with monitoring", "fair_budget": 1600},
    {"description": "Create a social media content calendar and 30 posts for LinkedIn and Instagram", "fair_budget": 350},
    {"description": "Implement Stripe payment integration in an existing Next.js e-commerce app", "fair_budget": 500},
    {"description": "Build a real-time chat application using Socket.io, React, and MongoDB", "fair_budget": 1100},
    {"description": "Conduct a UX audit of a mobile banking app and deliver an improvement report", "fair_budget": 600},
    {"description": "Create a Python automation script for invoice processing and QuickBooks sync", "fair_budget": 400},
    {"description": "Develop a Chrome extension for screenshot capture and annotation", "fair_budget": 350},
    {"description": "Build a recommendation engine using collaborative filtering for an e-commerce site", "fair_budget": 1800},
    {"description": "Design UI/UX for a 10-screen healthcare appointment booking app in Figma", "fair_budget": 1000},
    {"description": "Set up Elasticsearch with Kibana dashboards for application log monitoring", "fair_budget": 900},
    {"description": "Create a pitch deck template and full slide deck for a Series A fundraise", "fair_budget": 600},
    {"description": "Develop a custom WordPress plugin for membership management and subscriptions", "fair_budget": 800},
    {"description": "Build an ETL pipeline from MySQL to Snowflake with dbt transformations", "fair_budget": 2000},
    {"description": "Write unit and integration tests for a Python microservices architecture", "fair_budget": 700},
    {"description": "Create a React Native fitness tracking app with health API integration", "fair_budget": 3000},
    {"description": "Implement OAuth2 social login (Google, GitHub) for an existing web app", "fair_budget": 300},
    {"description": "Build a Tableau dashboard for sales performance with 8 KPI visualisations", "fair_budget": 850},
    {"description": "Create 20 product illustration icons for a fintech mobile app", "fair_budget": 500},
    {"description": "Develop a Python CLI tool for batch video transcription using Whisper", "fair_budget": 350},
    {"description": "Set up GitHub Actions CI/CD pipeline with staging and production environments", "fair_budget": 600},
    {"description": "Build a property listing website with search, filters, and map integration", "fair_budget": 1400},
    {"description": "Conduct penetration testing on a web application and deliver a security report", "fair_budget": 2000},
    {"description": "Build a customer support chatbot using Rasa and integrate it with Zendesk", "fair_budget": 1300},
    {"description": "Design email marketing templates (10) for a SaaS product onboarding sequence", "fair_budget": 400},
    {"description": "Build a Node.js GraphQL API with Apollo Server and MongoDB Atlas", "fair_budget": 1000},
    {"description": "Develop a Solidity smart contract for an NFT marketplace on Ethereum", "fair_budget": 2800},
    {"description": "Create a tutorial video series (5 videos, 10 min each) for a project management tool", "fair_budget": 700},
    {"description": "Implement multi-language support (i18n) for an existing React web application", "fair_budget": 500},
    {"description": "Build an automated testing suite using Selenium and pytest for a Django app", "fair_budget": 800},
    {"description": "Create interactive financial reporting visualisations using D3.js", "fair_budget": 1100},
    {"description": "Develop a Slack integration bot for HR leave management and approval workflows", "fair_budget": 650},
    {"description": "Write an SEO strategy report with full competitor and keyword analysis", "fair_budget": 550},
    {"description": "Build a containerised microservices architecture with Docker Compose", "fair_budget": 1900},
    {"description": "Create 10 animated social media ads in After Effects for a product launch", "fair_budget": 600},
]
print(f"Corpus: {len(GIG_CORPUS)} historical gigs")

In [ ]:
# Synthetic gig feed — mix of above-market and below-market posted budgets
GIG_FEED = [
    {"description": "Build a full-stack CRUD web app with React, Node.js, user auth, and role-based permissions", "posted_budget": 200, "url": "https://gigfeed.io/1"},
    {"description": "Develop a Python scraper for e-commerce prices across 5 sites with scheduling and Slack alerts", "posted_budget": 600, "url": "https://gigfeed.io/2"},
    {"description": "Design a modern brand identity: logo, colour palette, typography, and business card layout", "posted_budget": 180, "url": "https://gigfeed.io/3"},
    {"description": "Write 8 SEO-optimised blog posts for a B2B software company, 700–900 words each", "posted_budget": 560, "url": "https://gigfeed.io/4"},
    {"description": "Set up AWS infrastructure with VPC, RDS, S3, CloudFront, and a CI/CD pipeline using Terraform", "posted_budget": 2200, "url": "https://gigfeed.io/5"},
    {"description": "Build a Flutter mobile app for Android and iOS with Firebase auth and push notifications", "posted_budget": 1200, "url": "https://gigfeed.io/6"},
    {"description": "Create an animated product explainer video (90 seconds) with custom voiceover and motion graphics", "posted_budget": 1100, "url": "https://gigfeed.io/7"},
    {"description": "Implement Stripe subscription billing with webhooks in an existing Django web application", "posted_budget": 450, "url": "https://gigfeed.io/8"},
    {"description": "Build a Node.js GraphQL API with subscriptions, Redis caching, and PostgreSQL", "posted_budget": 1400, "url": "https://gigfeed.io/9"},
    {"description": "Design 5 Figma landing page mockups for a SaaS product with mobile-responsive variants", "posted_budget": 300, "url": "https://gigfeed.io/10"},
    {"description": "Configure a Kubernetes cluster on GCP with horizontal autoscaling and Prometheus monitoring", "posted_budget": 2000, "url": "https://gigfeed.io/11"},
    {"description": "Create 30 social media posts with graphics for LinkedIn and Instagram over one month", "posted_budget": 500, "url": "https://gigfeed.io/12"},
    {"description": "Build a web scraper and data extraction tool for a recruitment platform across 10 job boards", "posted_budget": 700, "url": "https://gigfeed.io/13"},
    {"description": "Develop a React Native iOS app with GPS tracking, offline sync, and push notifications", "posted_budget": 1000, "url": "https://gigfeed.io/14"},
    {"description": "Write unit and integration tests for a Python FastAPI microservices backend with full coverage", "posted_budget": 900, "url": "https://gigfeed.io/15"},
]
print(f"Feed: {len(GIG_FEED)} new gig postings")

In [ ]:
# Populate ChromaDB with the historical corpus (skip if already built)
DB_PATH = "gig_vectorstore"
ENCODER = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

db_client = chromadb.PersistentClient(path=DB_PATH)
collection = db_client.get_or_create_collection("gigs")

if collection.count() == 0:
    print("Populating ChromaDB ...")
    descriptions = [g["description"] for g in GIG_CORPUS]
    budgets      = [g["fair_budget"]   for g in GIG_CORPUS]
    vectors      = ENCODER.encode(descriptions).tolist()
    collection.add(
        documents=descriptions,
        embeddings=vectors,
        metadatas=[{"fair_budget": b} for b in budgets],
        ids=[f"gig_{i}" for i in range(len(descriptions))],
    )
    print(f"Added {collection.count()} gigs to ChromaDB")
else:
    print(f"ChromaDB already has {collection.count()} gigs — skipping rebuild")

## Agent Framework

Same base class as week 8 — coloured console logging so you can tell agents apart at a glance.

In [ ]:
class Agent:
    RED     = "\033[31m"
    GREEN   = "\033[32m"
    YELLOW  = "\033[33m"
    BLUE    = "\033[34m"
    CYAN    = "\033[36m"
    WHITE   = "\033[37m"
    BG_BLACK = "\033[40m"
    RESET   = "\033[0m"

    name: str  = ""
    color: str = WHITE

    def log(self, message: str):
        logging.info(f"{self.BG_BLACK}{self.color}[{self.name}] {message}{self.RESET}")

## Specialist Agent

In week 8 the specialist is a fine-tuned LLM deployed on Modal.  
Here we replicate that behaviour with a **cheap model + few-shot expert prompting** - the `SPECIALIST_FEW_SHOTS` in `prompts.py` are the labelled training examples; injecting them as chat history mimics what fine-tuning achieves at near-zero cost.

In [ ]:
class GigSpecialistAgent(Agent):
    name  = "Specialist Agent"
    color = Agent.RED

    def _build_messages(self, description: str) -> list:
        messages = [{"role": "system", "content": SPECIALIST_SYSTEM}]
        for gig_desc, rate in SPECIALIST_FEW_SHOTS:
            messages.append({"role": "user",      "content": gig_desc})
            messages.append({"role": "assistant", "content": rate})
        messages.append({"role": "user", "content": description})
        return messages

    def price(self, description: str) -> float:
        self.log("Estimating with few-shot specialist")
        response = client.chat.completions.create(
            model=MODEL,
            messages=self._build_messages(description),
            temperature=0.1,
        )
        result = extract_price(response.choices[0].message.content)
        self.log(f"Specialist estimate: ${result:.0f}")
        return result

In [ ]:
specialist = GigSpecialistAgent()
specialist.price("Build a Python REST API with PostgreSQL and JWT authentication")

## Frontier Agent

Uses RAG - queries ChromaDB for the 5 most similar historical gigs, then passes them as context to the LLM.  
Same pattern as `FrontierAgent` in week 8.

In [ ]:
class GigFrontierAgent(Agent):
    name  = "Frontier Agent"
    color = Agent.BLUE

    def find_similars(self, description: str):
        self.log("Searching ChromaDB for similar gigs")
        vector  = ENCODER.encode([description]).tolist()
        results = collection.query(query_embeddings=vector, n_results=5)
        docs    = results["documents"][0]
        budgets = [m["fair_budget"] for m in results["metadatas"][0]]
        return docs, budgets

    def price(self, description: str) -> float:
        docs, budgets = self.find_similars(description)
        self.log("Calling frontier model with RAG context")
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": FRONTIER_SYSTEM},
                {"role": "user",   "content": frontier_user(description, docs, budgets)},
            ],
            temperature=0.1,
        )
        result = extract_price(response.choices[0].message.content)
        self.log(f"Frontier estimate: ${result:.0f}")
        return result

In [ ]:
frontier = GigFrontierAgent()
frontier.price("Build a Python REST API with PostgreSQL and JWT authentication")

## Ensemble Agent

Combines both signals with a weighted average - frontier (RAG-grounded) gets more weight, specialist (few-shot) provides a quick baseline.

In [ ]:
class GigEnsembleAgent(Agent):
    name  = "Ensemble Agent"
    color = Agent.YELLOW

    def __init__(self):
        self.log("Initializing")
        self.specialist = GigSpecialistAgent()
        self.frontier   = GigFrontierAgent()
        self.log("Ready")

    def price(self, description: str) -> float:
        s = self.specialist.price(description)
        f = self.frontier.price(description)
        combined = f * 0.6 + s * 0.4
        self.log(f"Ensemble: specialist=${s:.0f}, frontier=${f:.0f} → combined=${combined:.0f}")
        return combined

In [ ]:
ensemble = GigEnsembleAgent()
ensemble.price("Build a Python REST API with PostgreSQL and JWT authentication")

## Scanner Agent

In week 8 the scanner fetches live RSS feeds and uses an LLM to extract clean descriptions from messy HTML.  
Here it selects the 5 most promising gigs from our synthetic feed using the same LLM-selection pattern.

In [ ]:
class GigScannerAgent(Agent):
    name  = "Scanner Agent"
    color = Agent.CYAN

    def _format_feed(self, feed: List[dict]) -> str:
        return "\n\n".join(
            f"Description: {g['description']}\nPosted Budget: ${g['posted_budget']}\nURL: {g['url']}"
            for g in feed
        )

    def scan(self, memory_urls: List[str] = []) -> List[dict]:
        feed = [g for g in GIG_FEED if g["url"] not in memory_urls]
        self.log(f"Scanning {len(feed)} unseen gigs from feed")
        if not feed:
            return []

        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SCANNER_SYSTEM},
                {"role": "user",   "content": SCANNER_USER_PREFIX + self._format_feed(feed) + SCANNER_USER_SUFFIX},
            ],
            response_format={"type": "json_object"},
            temperature=0.1,
        )
        data     = parse_json_safe(response.choices[0].message.content)
        selected = data.get("gigs", [])

        # Ensure posted_budget is present — fall back to feed data by URL match
        enriched = []
        for item in selected[:5]:
            url        = item.get("url", "")
            feed_match = next((g for g in feed if g["url"] == url), None)
            budget     = feed_match["posted_budget"] if feed_match else item.get("posted_budget", 0)
            enriched.append({"description": item.get("description", ""), "posted_budget": float(budget), "url": url})

        self.log(f"Scanner selected {len(enriched)} gigs")
        return enriched

In [ ]:
scanner = GigScannerAgent()
scanned = scanner.scan()
for g in scanned:
    print(f"${g['posted_budget']:>6.0f}  {g['description'][:70]}")

## Planner Agent

Orchestrates the full pipeline: scan → price each gig → rank by **value surplus** (posted budget − fair estimate).  
A positive surplus means the client is paying at or above market rate — the best opportunities for a freelancer.

In [ ]:
class GigOpportunity(BaseModel):
    description:    str
    posted_budget:  float
    fair_estimate:  float
    value_surplus:  float   # posted_budget - fair_estimate  (positive = above market)
    url:            str


class GigPlannerAgent(Agent):
    name  = "Planner Agent"
    color = Agent.GREEN
    SURPLUS_THRESHOLD = 50   # only surface gigs paying $50+ above fair market

    def __init__(self):
        self.log("Initializing")
        self.scanner  = GigScannerAgent()
        self.ensemble = GigEnsembleAgent()
        self.log("Ready")

    def _evaluate(self, gig: dict) -> GigOpportunity:
        estimate = self.ensemble.price(gig["description"])
        surplus  = gig["posted_budget"] - estimate
        return GigOpportunity(
            description=gig["description"],
            posted_budget=gig["posted_budget"],
            fair_estimate=round(estimate, 2),
            value_surplus=round(surplus, 2),
            url=gig["url"],
        )

    def plan(self, memory_urls: List[str] = []) -> List[GigOpportunity]:
        self.log("Starting planning run")
        gigs = self.scanner.scan(memory_urls=memory_urls)
        if not gigs:
            self.log("No new gigs found")
            return []
        opportunities = [self._evaluate(g) for g in gigs]
        opportunities.sort(key=lambda o: o.value_surplus, reverse=True)
        best = opportunities[0]
        self.log(f"Best opportunity: surplus=${best.value_surplus:.0f} — '{best.description[:55]}...'")
        return opportunities

In [ ]:
planner     = GigPlannerAgent()
opportunities = planner.plan()

print(f"\n{'Description':<60} {'Posted':>8} {'Estimate':>9} {'Surplus':>8}")
print("-" * 90)
for opp in opportunities:
    flag = " ✓" if opp.value_surplus >= planner.SURPLUS_THRESHOLD else ""
    print(f"{opp.description[:58]:<60} ${opp.posted_budget:>6.0f}   ${opp.fair_estimate:>6.0f}   ${opp.value_surplus:>+6.0f}{flag}")

## Gradio UI

Click **"Scan for Opportunities"** to run the full pipeline and populate the table.  
Each run skips already-seen gigs - hit the button again to process the next batch.

In [ ]:
agent = GigPlannerAgent()

def to_row(opp: GigOpportunity):
    surplus_str = f"${opp.value_surplus:+.0f}"
    return [opp.description[:80], f"${opp.posted_budget:.0f}", f"${opp.fair_estimate:.0f}", surplus_str, opp.url]

with gr.Blocks(title="Gig Deal Hunter", fill_width=True) as ui:
    seen_urls = gr.State([])

    with gr.Row():
        gr.Markdown('<div style="text-align:center;font-size:26px"><b>Gig Deal Hunter</b></div>')
    with gr.Row():
        gr.Markdown('<div style="text-align:center;font-size:14px;color:#888">Surfaces freelance gigs where the posted budget exceeds the fair market rate - positive surplus = above-market pay</div>')
    with gr.Row():
        scan_btn = gr.Button("Scan for Opportunities", variant="primary", scale=1)
    with gr.Row():
        opportunities_df = gr.Dataframe(
            headers=["Description", "Posted Budget", "Fair Estimate", "Surplus", "URL"],
            wrap=True,
            column_widths=[5, 1, 1, 1, 2],
            max_height=450,
        )

    def run_scan(seen):
        new_opps = agent.plan(memory_urls=seen)
        new_urls = seen + [o.url for o in new_opps]
        rows     = [to_row(o) for o in new_opps]
        return new_urls, rows

    scan_btn.click(run_scan, inputs=[seen_urls], outputs=[seen_urls, opportunities_df])

ui.launch(inbrowser=True)